In [1]:
pip install pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 28.0 MB/s  0:00:00eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 37.3 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [pandas]2m1/2 [pandas]
Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd

# Extração:

In [11]:
# Atualizar
ano2caminho = {
    '2014': "data/2014/Acidentes-2014.csv"
}

l_df = []

for ano, cam in ano2caminho.items():
    try:
        df =pd.read_csv(cam, encoding='utf-8', sep=';')
        l_df.append(df)
    except Exception as e:
        print(f"Erro no ano {ano}: {e}")

df = pd.concat(l_df)
print(f"Total linhas da tabela unificada: {len(df)}")

Total linhas da tabela unificada: 1734


# Transformação:

In [4]:
display(df.info())
print(f"Valores null:\n{df.isnull().sum()}")
df.nunique(dropna=False) # Considera valores nulos

<class 'pandas.DataFrame'>
RangeIndex: 1734 entries, 0 to 1733
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   data       1734 non-null   str    
 1   tipo       1734 non-null   str    
 2   detalhes   1734 non-null   str    
 3   longitude  1734 non-null   float64
 4   latitude   1734 non-null   float64
dtypes: float64(2), str(3)
memory usage: 67.9 KB


None

Valores null:
data         0
tipo         0
detalhes     0
longitude    0
latitude     0
dtype: int64


data          286
tipo           14
detalhes       12
longitude    1669
latitude     1694
dtype: int64

### Nota:
Ano de 2014 não possui valores nulos para nenhuma coluna, mas ainda apresenta ausência de algumas colunas presentes em outros anos que potencialmente introduziriam colunas nulas em linhas de 2014.

In [6]:
print(f"Valores possíveis da coluna 'detalhes':\n{df["detalhes"].unique()}\n")
print(f"Valores possíveis da coluna 'tipo':\n{df["tipo"].unique()}\n")

Valores possíveis da coluna 'detalhes':
<StringArray>
[               'Moto',            'Ciclista',            'Pedestre',
 'Acidente com animal',            'Caminhão',                'Taxi',
              'Onibus',          'Ciclomotor',              'Ônibus',
  'Acidente com morte',                'Táxi',             'Carroça']
Length: 12, dtype: str

Valores possíveis da coluna 'tipo':
<StringArray>
[            'Colisoes',            'Ciclistas',       'Atropelamentos',
    'Moto e Ciclomotor', 'Ciclistas e Pedestre',  'Automóveis e outros',
 'Ciclistas e pedestre', 'Motos e Ciclomotores', 'Pedestres e ciclista',
         'Motocicletas',            'Pedestres',         'Ciclomotores',
           'Automóveis',               'Outros']
Length: 14, dtype: str



### Tratamento das colunas

Em detalhes vamos agrupar: 
"Táxi" e "Taxi" como "taxi", além de "Onibus" e "Ônibus" como "onibus".

Em tipo vamos agrupar: 
"Moto e Ciclomotor" e "Motos e Ciclomotores" como "motos e ciclomotores". Além de "Ciclistas e Pedestre", "Ciclistas e pedestre", "Pedestres e ciclista" como "ciclistas e pedestres". "Automóveis" e "Automóveis e outros" como "automoveis (diversos)".

Além disso, vamos remover acentuações, deixar tudo minúsculo.
Os valores de tipo devem estar no plural, os valores de detalhes no singular.


In [13]:
import unicodedata

def normalizar_texto(texto):
    if pd.isna(texto):
        return ""
    
    texto = str(texto).strip().lower()
    
    texto = ''.join(c for c in unicodedata.normalize('NFD', texto) if unicodedata.category(c) != 'Mn')
    return texto

def normalizar_tipo(texto):
    texto = normalizar_texto(texto)
    dicionario_padrao = {
            "moto e ciclomotor": "motos e ciclomotores",
            "ciclistas e pedestre": "ciclistas e pedestres",
            "pedestres e ciclista": "ciclistas e pedestres",
            "automoveis": "automoveis (diversos)",
            "automoveis e outros": "automoveis (diversos)"
            }

    return dicionario_padrao.get(texto, texto)

def normalizar_detalhes(texto):
    texto = normalizar_texto(texto)
    dicionario_padrao = {}

    return dicionario_padrao.get(texto, texto)

df['tipo'] = df['tipo'].apply(normalizar_tipo)
df['detalhes'] = df['detalhes'].apply(normalizar_detalhes)

print(f"Valores possíveis da coluna 'tipo':\n{df["tipo"].unique()}\n")
print(f"Valores possíveis da coluna 'detalhes':\n{df["detalhes"].unique()}\n")

Valores possíveis da coluna 'tipo':
<StringArray>
[             'colisoes',             'ciclistas',        'atropelamentos',
  'motos e ciclomotores', 'ciclistas e pedestres', 'automoveis (diversos)',
          'motocicletas',             'pedestres',          'ciclomotores',
                'outros']
Length: 10, dtype: str

Valores possíveis da coluna 'detalhes':
<StringArray>
[               'moto',            'ciclista',            'pedestre',
 'acidente com animal',            'caminhao',                'taxi',
              'onibus',          'ciclomotor',  'acidente com morte',
             'carroca']
Length: 10, dtype: str

